In [1]:
# Celda A1 - Configuracion, rutas y descarga del directorio oficial
import requests
import pandas as pd
import numpy as np
import re
from pathlib import Path
from datetime import date

HOY = date.today().isoformat()

BASE = Path('/Users/ppizam/Claude/Master Thesis/Desarrollo/Metodologia/Lista Maestra de Tickers')
V2 = BASE / 'Lista maestra V2'
SNAP_DIR = BASE / 'snapshots_directorio'
SNAP_DIR.mkdir(exist_ok=True)

URLS = {
    'nasdaqlisted': 'https://www.nasdaqtrader.com/dynamic/SymDir/nasdaqlisted.txt',
    'otherlisted':  'https://www.nasdaqtrader.com/dynamic/SymDir/otherlisted.txt',
}

# Descarga y guardado FECHADO (primer snapshot mensual de la practica nueva)
rutas = {}
for nombre, url in URLS.items():
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    destino = SNAP_DIR / f'{nombre}_{HOY}.txt'
    destino.write_text(r.text)
    rutas[nombre] = destino
    print(nombre, '->', destino.name, '|', len(r.text.splitlines()), 'lineas')
# Esperado: nasdaqlisted ~4,000-5,500 lineas y otherlisted ~6,000-8,000
# (no tengo el numero exacto de hoy; lo registramos como referencia del snapshot)
# Si la descarga fallara, tu build_master_list.py ya baja estos mismos archivos: usalo como plan B

nasdaqlisted -> nasdaqlisted_2026-07-17.txt | 5567 lineas
otherlisted -> otherlisted_2026-07-17.txt | 7494 lineas


In [2]:
# Celda A2 - Parseo y normalizacion de simbolos
def leer_pipe(path):
    df = pd.read_csv(path, sep='|', dtype=str)
    # La ultima fila es el pie "File Creation Time..."
    df = df[~df.iloc[:, 0].astype(str).str.startswith('File Creation')]
    return df

nas = leer_pipe(rutas['nasdaqlisted'])
oth = leer_pipe(rutas['otherlisted'])
print('nasdaqlisted cols:', list(nas.columns))
print('otherlisted  cols:', list(oth.columns))

# Bolsas de otherlisted: A = NYSE American, N = NYSE, P = NYSE Arca,
# Z = Cboe BZX, V = IEX (verificar contra las columnas impresas arriba)
MAPA_BOLSA = {'A': 'NYSE American', 'N': 'NYSE', 'P': 'NYSE Arca', 'Z': 'Cboe BZX', 'V': 'IEX'}

# base_symbol: recorta sufijos de clase/preferente/warrant para cruzar contra
# el ticker "plano" de CRSP. HEURISTICA documentada: corta en el primer
# separador (. - $ + = /) o en la primera minuscula (notacion p de preferentes)
def base_symbol(s):
    if not isinstance(s, str):
        return ''
    s = s.strip()
    corte = re.split(r'[.\-$+=/ ]', s)[0]
    m = re.match(r'^[A-Z0-9]+', corte)
    return m.group(0) if m else corte

directorio = pd.concat([
    pd.DataFrame({
        'symbol_raw': nas['Symbol'],
        'security_name': nas['Security Name'],
        'exchange': 'NASDAQ',
        'market_category': nas['Market Category'],
        'financial_status': nas.get('Financial Status'),
        'is_etf_dir': nas['ETF'].eq('Y'),
        'is_test_issue': nas['Test Issue'].eq('Y'),
    }),
    pd.DataFrame({
        'symbol_raw': oth['ACT Symbol'],
        'security_name': oth['Security Name'],
        'exchange': oth['Exchange'].map(MAPA_BOLSA).fillna(oth['Exchange']),
        'market_category': '',
        'financial_status': '',
        'is_etf_dir': oth['ETF'].eq('Y'),
        'is_test_issue': oth['Test Issue'].eq('Y'),
    }),
], ignore_index=True)

directorio = directorio[~directorio['is_test_issue']]
directorio = directorio.assign(base=directorio['symbol_raw'].map(base_symbol))
print(len(directorio), 'valores en el directorio (sin test issues)')
print(directorio['exchange'].value_counts().to_string())

nasdaqlisted cols: ['Symbol', 'Security Name', 'Market Category', 'Test Issue', 'Financial Status', 'Round Lot Size', 'ETF', 'NextShares']
otherlisted  cols: ['ACT Symbol', 'Security Name', 'Exchange', 'CQS Symbol', 'ETF', 'Round Lot Size', 'Test Issue', 'NASDAQ Symbol']
13024 valores en el directorio (sin test issues)
exchange
NASDAQ           5557
NYSE             2921
NYSE Arca        2692
Cboe BZX         1545
NYSE American     309


In [3]:
# Celda A3 - Cargar el padron y poblar los campos del directorio en activos
vig = pd.read_csv(V2 / 'padron_vigencias_2020_2024_mkt.csv',
                  keep_default_na=False, na_values=[''],
                  parse_dates=['fecha_inicio', 'fecha_fin'])
print(len(vig), vig['permno'].nunique())   # debe dar 12961 11899

activos = vig[vig['estado_vida'] == 'active'].copy()   # 8,900 vidas abiertas al 31-dic-2024

# Cruce por simbolo base; nos quedamos con una fila del directorio por base
# (preferimos la accion comun: la de nombre mas corto suele ser la comun)
dir_1 = (directorio.sort_values('symbol_raw', key=lambda s: s.str.len())
                   .drop_duplicates('base', keep='first'))

cruce = activos.merge(
    dir_1[['base', 'market_category', 'financial_status', 'is_etf_dir', 'exchange']],
    left_on='ticker', right_on='base', how='left', suffixes=('', '_dir'))

print('activos que cruzan con el directorio de hoy:',
      round(cruce['base'].notna().mean(), 3))
# Interpretacion: los que NO cruzan son candidatos a baja o renombre 2025-2026
# (se listan en la celda A5); NO es un error de cruce

# Guardar los campos del directorio por llave (permno, bloque) para integrarlos al padron
campos_dir = cruce[cruce['base'].notna()][
    ['permno', 'bloque', 'market_category', 'financial_status']]
campos_dir.to_csv(V2 / f'campos_directorio_{HOY}.csv', index=False)
print(len(campos_dir), 'filas guardadas en', f'campos_directorio_{HOY}.csv')

12961 11899
activos que cruzan con el directorio de hoy: 0.874
7778 filas guardadas en campos_directorio_2026-07-17.csv


In [4]:
# Celda A4 - Diff 1: candidatos a ALTA 2025 - jun 2026
# Simbolos del directorio de hoy (sin ETFs) cuyo base no corresponde a ninguna vida abierta
tickers_abiertos = set(activos['ticker'])

altas = directorio[
    (~directorio['is_etf_dir'])
    & (~directorio['base'].isin(tickers_abiertos))
].copy()

# Colapsar por base (clases/preferentes del mismo emisor cuentan una vez)
altas_u = altas.drop_duplicates('base')
print(len(altas_u), 'candidatos a alta (emisores nuevos o renombres 2025-2026)')
print(altas_u[['symbol_raw', 'security_name', 'exchange']].head(25).to_string(index=False))

# Verificacion con los sucesores 2025 que la auditoria ya confirmo:
SUCESORES_2025 = ['XYZ', 'SGI', 'XIFR', 'JBTM', 'ONC', 'TBCH', 'AAMI', 'HTO',
                  'PRSU', 'DVLT', 'JOYY', 'NAGE', 'IMDX', 'VIVS', 'RNTX',
                  'RDGT', 'CBIO', 'HTB', 'AMBR', 'ACCS']
en_altas = [t for t in SUCESORES_2025 if t in set(altas_u['base'])]
print('sucesores 2025 detectados como alta:', len(en_altas), 'de 20 ->', en_altas)
# Esperado: la gran mayoria de los 20 (si alguno falta, puede haber cambiado otra vez o salido de bolsa)

altas_u.to_csv(V2 / f'candidatos_altas_2025_2026_{HOY}.csv', index=False)

ERROR! Session/line number was not unique in database. History logging moved to new session 19
2011 candidatos a alta (emisores nuevos o renombres 2025-2026)
symbol_raw                                                     security_name exchange
      AACB              Artius II Acquisition Inc. - Class A Ordinary Shares   NASDAQ
     AACBR                               Artius II Acquisition Inc. - Rights   NASDAQ
     AACBU                                Artius II Acquisition Inc. - Units   NASDAQ
      AACI             Armada Acquisition Corp. III - Class A Ordinary Share   NASDAQ
     AACIU                              Armada Acquisition Corp. III - Units   NASDAQ
     AACIW                            Armada Acquisition Corp. III - Warrant   NASDAQ
      AACO                Abony Acquisition Corp. I - Class A Ordinary Share   NASDAQ
     AACOU                                 Abony Acquisition Corp. I - Units   NASDAQ
     AACOW                              Abony Acquisition Corp. I - 

In [5]:
# Celda A5 - Diff 2: candidatos a BAJA o RENOMBRE 2025 - jun 2026
# Vidas abiertas al 31-dic-2024 cuyo ticker ya no aparece en el directorio de hoy
bases_directorio = set(directorio['base'])

bajas_cand = activos[~activos['ticker'].isin(bases_directorio)].copy()
print(len(bajas_cand), 'vidas abiertas cuyo simbolo ya no esta listado hoy')

# Verificacion: los renombres 2025 verificados deben aparecer aqui por su ticker VIEJO
VIEJOS_2025 = ['SQ', 'TPX', 'NEP', 'JBT', 'BGNE', 'HEAR', 'BSIG', 'SJW', 'VVI',
               'WISA', 'YY', 'CDXC', 'OCX', 'ONVO', 'ALRN', 'CJJD', 'GLYC',
               'HTBI', 'ICLK', 'ISDR']
en_bajas = [t for t in VIEJOS_2025 if t in set(bajas_cand['ticker'])]
print('tickers viejos 2025 detectados:', len(en_bajas), 'de 20 ->', en_bajas)
# Tambien deben aparecer las 4 bajas reales verificadas: PLYA, MRIN, SBT, BROG
print('bajas reales verificadas presentes:',
      [t for t in ['PLYA', 'MRIN', 'SBT', 'BROG'] if t in set(bajas_cand['ticker'])])

cols_exp = ['permno', 'bloque', 'ticker', 'comnam', 'cik', 'exchcd',
            'shrcd', 'market_cap_usd', 'fecha_fin']
bajas_cand[cols_exp].to_csv(V2 / f'candidatos_bajas_2025_2026_{HOY}.csv', index=False)
print('guardado en', f'candidatos_bajas_2025_2026_{HOY}.csv')

1122 vidas abiertas cuyo simbolo ya no esta listado hoy
tickers viejos 2025 detectados: 20 de 20 -> ['SQ', 'TPX', 'NEP', 'JBT', 'BGNE', 'HEAR', 'BSIG', 'SJW', 'VVI', 'WISA', 'YY', 'CDXC', 'OCX', 'ONVO', 'ALRN', 'CJJD', 'GLYC', 'HTBI', 'ICLK', 'ISDR']
bajas reales verificadas presentes: ['PLYA', 'MRIN', 'SBT', 'BROG']
guardado en candidatos_bajas_2025_2026_2026-07-17.csv


In [6]:
# Celda A6 - Resumen de la fase A
print('Snapshot del directorio guardado:', HOY)
print('Campos de directorio poblados para activos que siguen listados')
print('Candidatos a alta 2025-2026:', len(altas_u))
print('Candidatos a baja/renombre 2025-2026:', len(bajas_cand))
print()
print('Siguiente: Fase C (flags) sobre el mismo notebook, y Fase D con estas dos listas')

Snapshot del directorio guardado: 2026-07-17
Campos de directorio poblados para activos que siguen listados
Candidatos a alta 2025-2026: 2011
Candidatos a baja/renombre 2025-2026: 1122

Siguiente: Fase C (flags) sobre el mismo notebook, y Fase D con estas dos listas


In [ ]:
# Celda C0 - Recarga (correr siempre que el kernel sea nuevo)
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import date

HOY = date.today().isoformat()
V2 = Path('/Users/ppizam/Claude/Master Thesis/Desarrollo/Metodologia/Lista Maestra de Tickers/Lista maestra V2')

vig = pd.read_csv(V2 / 'padron_vigencias_2020_2024_mkt.csv',
                  keep_default_na=False, na_values=[''],
                  parse_dates=['fecha_inicio', 'fecha_fin'])
print(len(vig), vig['permno'].nunique())   # debe dar 12961 11899

In [ ]:
# Celda C1 - Flag is_spac (heuristica de nombre, documentada; marcar, no excluir)
# Patron: "ACQUISITION" como palabra en el nombre CRSP (comnam) o del directorio
pat_spac = r'\bACQUISITION\b|\bACQUISITIONS\b|\bSPAC\b'

vig = vig.assign(
    is_spac=vig['comnam'].str.upper().str.contains(pat_spac, regex=True, na=False)
)
print(vig['is_spac'].sum(), 'vidas marcadas como SPAC')
print(vig[vig['is_spac']]['comnam'].drop_duplicates().head(15).to_string(index=False))
# Revisa la muestra: deben ser Acquisition Corps; documentar cualquier falso positivo detectado

In [ ]:
# Celda C2 - Flag is_crypto_equity (v3: excluye todo shrcd 7x, ETFs y trusts ETP)
LISTA_CRYPTO = {'RIOT', 'MARA', 'HIVE', 'HVBT', 'HUT', 'BITF', 'CLSK', 'CIFR',
                'WULF', 'CORZ', 'IREN', 'BTBT', 'CAN', 'EBON', 'SOS', 'BTCS',
                'GREE', 'ARBK', 'COIN', 'MSTR', 'BTDR', 'ABTC', 'BTM',
                'SDIG', 'DGHI', 'MIGI', 'BTOG'}
# Caso limite con decision documentada: 'APLD' (ex Applied Blockchain, hoy datacenters IA)

pat_crypto = r'\bBITCOIN\b|\bBLOCKCHAIN\b|\bCRYPTO\b|\bETHEREUM\b|\bDIGITAL ASSET'

por_lista = vig['ticker'].isin(LISTA_CRYPTO)
por_nombre = vig['comnam'].str.upper().str.contains(pat_crypto, regex=True, na=False)
es_fondo = (vig['shrcd'] // 10) == 7   # ETFs (73), trusts ETP (74) y units

vig = vig.assign(is_crypto_equity=(por_lista | por_nombre) & ~es_fondo)
print(vig['is_crypto_equity'].sum(), 'vidas marcadas como cripto-equity')
print(vig[vig['is_crypto_equity']][['ticker', 'comnam', 'shrcd']]
      .drop_duplicates().to_string(index=False))
# Esperado: ~32 vidas, todas con shrcd 11/12/31 (empresas operadoras)

In [ ]:
# Celda C3 - Guardar el padron con flags
vig.to_csv(V2 / 'padron_vigencias_2020_2024_mkt_flags.csv', index=False)
print(len(vig.columns), 'columnas |', len(vig), 'filas')

In [7]:
# Celda D1 - Agrupar los candidatos a alta por emisor real
# NASDAQ pega sufijos de units (U), warrants (W) y rights (R) sin separador:
# AACBU/AACBW/AACBR son el mismo emisor que AACB
altas = pd.read_csv(V2 / 'candidatos_altas_2025_2026_2026-07-17.csv')

def raiz(s):
    if isinstance(s, str) and len(s) == 5 and s[-1] in 'UWR':
        return s[:4]
    return s

altas = altas.assign(raiz=altas['base'].map(raiz))
bases = set(altas['base'])
altas = altas.assign(es_derivado=(altas['base'] != altas['raiz']) & altas['raiz'].isin(bases))

emisores = altas[~altas['es_derivado']].drop_duplicates('raiz').copy()

# Excluir los 20 sucesores de renombres 2025: NO son altas nuevas,
# se integran como baja + alta enlazadas desde eventos_2025_verificados.csv
eventos = pd.read_csv(V2 / 'eventos_2025_verificados.csv')
sucesores = set(eventos.loc[eventos['tipo'] == 'renombre', 'ticker_nuevo'])
emisores = emisores[~emisores['raiz'].isin(sucesores)]

print(len(emisores), 'emisores nuevos reales por fechar (altas 2025 - jul 2026)')
print(emisores.groupby('exchange').size().to_string())
print()
print(emisores[['symbol_raw', 'security_name', 'exchange']].head(30).to_string(index=False))

emisores.to_csv(V2 / 'altas_emisores_2025_2026.csv', index=False)

1537 emisores nuevos reales por fechar (altas 2025 - jul 2026)
exchange
Cboe BZX            4
NASDAQ           1139
NYSE              331
NYSE American      50
NYSE Arca          13

symbol_raw                                                                                                                                 security_name exchange
      AACB                                                                                          Artius II Acquisition Inc. - Class A Ordinary Shares   NASDAQ
      AACI                                                                                         Armada Acquisition Corp. III - Class A Ordinary Share   NASDAQ
      AACO                                                                                            Abony Acquisition Corp. I - Class A Ordinary Share   NASDAQ
      AACP                                                                                             Apogee Acquisition Corp - Class A Ordinary Shares   NASDAQ
      A

In [8]:
# Celda D2 - Depurar altas: solo acciones (comunes/ADRs), fuera derivados y preferentes
emisores = pd.read_csv(V2 / 'altas_emisores_2025_2026.csv')

# 1. Filtro por descripcion del instrumento (el directorio lo dice textualmente)
pat_no_accion = (r'Warrant|Right|Unit|Preferred|Preference|Depositary|Notes? Due|'
                 r'Debenture|Bond|Trust Preferred|Subordinated')
es_no_accion = emisores['security_name'].str.contains(pat_no_accion, case=False, regex=True, na=False)

# 2. Raiz de 5ta letra contra el padron: si el simbolo es de 5 letras y su raiz
#    de 4 ya existe como vida abierta del padron, es un derivado del mismo emisor
tickers_abiertos = set(vig.loc[vig['estado_vida'] == 'active', 'ticker'])
raiz4 = emisores['raiz'].astype(str).str[:4]
es_derivado_padron = (emisores['raiz'].str.len() == 5) & raiz4.isin(tickers_abiertos)

acciones = emisores[~es_no_accion & ~es_derivado_padron].copy()
print(len(emisores), '->', len(acciones), 'emisores accion tras depurar')
print(acciones.groupby('exchange').size().to_string())
print()
print(acciones[['symbol_raw', 'security_name', 'exchange']].head(30).to_string(index=False))

acciones.to_csv(V2 / 'altas_emisores_2025_2026_depurado.csv', index=False)

1537 -> 926 emisores accion tras depurar
exchange
Cboe BZX           4
NASDAQ           682
NYSE             185
NYSE American     43
NYSE Arca         12

symbol_raw                                                       security_name exchange
      AACB                Artius II Acquisition Inc. - Class A Ordinary Shares   NASDAQ
      AACI               Armada Acquisition Corp. III - Class A Ordinary Share   NASDAQ
      AACO                  Abony Acquisition Corp. I - Class A Ordinary Share   NASDAQ
      AACP                   Apogee Acquisition Corp - Class A Ordinary Shares   NASDAQ
      AAPG   Ascentage Pharma Group International - American Depository Shares   NASDAQ
      AARD                          Aardvark Therapeutics, Inc. - Common Stock   NASDAQ
      ABTC                       American Bitcoin Corp. - Class A Common Stock   NASDAQ
      ACAA          Averin Capital Acquisition Corp. - Class A Ordinary Shares   NASDAQ
      ACCL               Acco Group Holdings Limited

In [9]:
# Celda D3 - CIK de cada alta via el archivo oficial de la SEC
import requests, time

# La SEC exige identificarse en el User-Agent (uso academico)
UA = {'User-Agent': 'Pedro Piza pedro.piza@gmail.com (investigacion academica ITAM)'}

r = requests.get('https://www.sec.gov/files/company_tickers_exchange.json',
                 headers=UA, timeout=60)
r.raise_for_status()
sec = pd.DataFrame(r.json()['data'], columns=r.json()['fields'])
print(len(sec), 'registros SEC |', list(sec.columns))
# columnas esperadas: cik, name, ticker, exchange

acciones = pd.read_csv(V2 / 'altas_emisores_2025_2026_depurado.csv')
sec_map = sec.drop_duplicates('ticker').set_index('ticker')

acciones = acciones.assign(
    cik=acciones['raiz'].map(sec_map['cik']),
    nombre_sec=acciones['raiz'].map(sec_map['name']),
)
print('con CIK:', acciones['cik'].notna().sum(), 'de', len(acciones),
      '| sin CIK:', acciones['cik'].isna().sum())
# Los sin CIK suelen ser listados muy recientes o simbologia distinta; quedan
# para revision manual, deberian ser pocos

10426 registros SEC | ['cik', 'name', 'ticker', 'exchange']
con CIK: 924 de 926 | sin CIK: 2


In [10]:
# Celda D4 - Fecha de alta aproximada: primer formulario 8-A en EDGAR por CIK
# (v2: corregido el manejo de los CIK faltantes)
CHECKPOINT_D = V2 / 'altas_fechadas_edgar.csv'

hechos = set()
filas = []
if CHECKPOINT_D.exists():
    prev = pd.read_csv(CHECKPOINT_D)
    hechos = set(prev['cik'].dropna().astype(int))
    filas = prev.to_dict('records')
    print('reanudando:', len(hechos))

con_cik = acciones[acciones['cik'].notna()].copy()
con_cik['cik'] = con_cik['cik'].astype(int)
pend = con_cik[~con_cik['cik'].isin(hechos)]
print(len(pend), 'CIKs por consultar')

# De paso: las altas SIN cik, para revision manual
print(acciones[acciones['cik'].isna()][['symbol_raw', 'security_name', 'exchange']]
      .to_string(index=False))

for i, row in enumerate(pend.itertuples(), 1):
    cik = int(row.cik)
    fila = {'ticker': row.raiz, 'cik': cik, 'fecha_8a': None,
            'form_8a': None, 'primer_filing': None, 'error': ''}
    try:
        url = f'https://data.sec.gov/submissions/CIK{cik:010d}.json'
        j = requests.get(url, headers=UA, timeout=30).json()
        forms = j['filings']['recent']['form']
        fechas = j['filings']['recent']['filingDate']
        ochoa = [(f, d) for f, d in zip(forms, fechas) if f.startswith('8-A')]
        if ochoa:
            f, d = min(ochoa, key=lambda x: x[1])
            fila['fecha_8a'], fila['form_8a'] = d, f
        fila['primer_filing'] = min(fechas) if fechas else None
    except Exception as e:
        fila['error'] = str(e)[:100]
    filas.append(fila)
    if i % 50 == 0 or i == len(pend):
        pd.DataFrame(filas).to_csv(CHECKPOINT_D, index=False)
        print(f'{i}/{len(pend)} guardado')
    time.sleep(0.15)

res = pd.DataFrame(filas)
print('con fecha 8-A:', res['fecha_8a'].notna().sum(), 'de', len(res))
print('errores:', (res['error'].fillna('') != '').sum())

reanudando: 911
0 CIKs por consultar
symbol_raw                                                                               security_name exchange
       DPU                                                   Top KingWin Ltd - Class A Ordinary Shares   NASDAQ
     MER$K Bank of America Corporation Income Capital Obligation Notes initially due December 15, 2066     NYSE
con fecha 8-A: 830 de 924
errores: 0


In [11]:
# Celda D5 - Consolidar la fecha de alta de cada emisor nuevo
ed = pd.read_csv(V2 / 'altas_fechadas_edgar.csv',
                 parse_dates=['fecha_8a', 'primer_filing'])

INICIO_COLA = pd.Timestamp('2025-01-01')

def clasifica(r):
    # 8-A reciente = registro de listado 2025-2026: fecha confiable
    if pd.notna(r['fecha_8a']) and r['fecha_8a'] >= INICIO_COLA:
        return pd.Series([r['fecha_8a'], 'edgar_8a'])
    # 8-A viejo = empresa que ya cotizaba antes (relistado, uplisting de OTC,
    # o simbologia que no cruzo con el padron): a revision
    if pd.notna(r['fecha_8a']):
        return pd.Series([r['fecha_8a'], 'revisar_8a_viejo'])
    # Sin 8-A pero primer filing reciente: entidad nueva, el filing acota la fecha
    if pd.notna(r['primer_filing']) and r['primer_filing'] >= INICIO_COLA:
        return pd.Series([r['primer_filing'], 'edgar_primer_filing'])
    return pd.Series([pd.NaT, 'manual'])

ed[['fecha_alta_aprox', 'metodo_fecha']] = ed.apply(clasifica, axis=1)
print(ed['metodo_fecha'].value_counts().to_string())
print()
print('revisar_8a_viejo (muestra):')
print(ed[ed['metodo_fecha'] == 'revisar_8a_viejo'][['ticker', 'fecha_8a', 'primer_filing']]
      .head(20).to_string(index=False))

ed.to_csv(V2 / 'altas_fechadas_edgar.csv', index=False)

metodo_fecha
edgar_8a               610
revisar_8a_viejo       220
manual                  63
edgar_primer_filing     31

revisar_8a_viejo (muestra):
ticker   fecha_8a primer_filing
  ACFN 2007-12-13    2003-04-22
  ADAM 2008-06-03    2008-02-25
  AGPU 2014-12-17    2014-08-08
  AIFA 2017-10-02    2017-06-09
  AIIO 2022-11-14    2022-06-07
  AIOS 2016-07-12    2014-03-31
  AIXC 2015-06-15    2009-03-17
  ALOY 2021-11-09    2013-11-27
   ALP 2021-02-18    2000-08-01
  AMCI 2022-12-16    2022-07-28
  ASBP 2022-02-17    2021-03-16
  ASPC 2024-11-07    2021-11-10
   AUC 2019-04-18    2018-10-15
  AURE 2023-06-29    2019-02-15
   AVX 2021-07-02    2020-10-13
   AXG 2023-08-09    2022-12-23
  AZIO 2017-04-28    2012-12-03
  BBOT 2024-02-08    2023-12-20
  BCIC 2008-03-31    2006-12-19
  BGDE 2021-09-28    2003-02-18


In [13]:
# Celda D5b - Subclasificar los 220 'revisar_8a_viejo' y los 63 'manual'
pendientes_fecha = ed[ed['metodo_fecha'].isin(['revisar_8a_viejo', 'manual'])].copy()

# tickers con alguna vida en el padron (cualquier estado)
vidas_por_ticker = vig.groupby('ticker')['estado_vida'].apply(set)
def perfil(tk):
    estados = vidas_por_ticker.get(tk)
    if estados is None:
        return 'nuevo_o_uplisting_otc'
    if 'delisted' in estados or 'renombrada' in estados:
        return 'posible_relistado'
    return 'revisar_cruce'   # existe como active: no debio estar en altas

pendientes_fecha['perfil'] = pendientes_fecha['ticker'].map(perfil)
print(pendientes_fecha['perfil'].value_counts().to_string())
print()
print(pendientes_fecha[pendientes_fecha['perfil'] == 'posible_relistado']
      [['ticker', 'fecha_8a', 'primer_filing']].head(15).to_string(index=False))

perfil
nuevo_o_uplisting_otc    251
posible_relistado         32

ticker   fecha_8a primer_filing
  AMCI 2022-12-16    2022-07-28
  ASPC 2024-11-07    2021-11-10
   AVX 2021-07-02    2020-10-13
   CCC 2020-08-13    2020-07-17
    CD 2015-03-26    2014-08-08
  CHAI 2020-09-24    2015-08-04
  CIRC 2021-08-25    2014-06-10
  DBGI 2021-05-11    2016-03-23
  DFNS 2020-06-22    2019-11-26
   DOO 2018-09-11    2018-09-11
  ECHO        NaT    2011-01-04
  ELOX 2018-04-24    2008-11-21
  FISV 2019-07-01    2019-06-24
   FLD 2021-12-14    2021-10-21
  MRLN 2024-10-31    2024-07-12


In [15]:
# Celda D6 - Clasificar los candidatos a baja restantes via CIK (SEC + Form 25)
bc = pd.read_csv(sorted(V2.glob('candidatos_bajas_2025_2026_*.csv'))[-1])
eventos = pd.read_csv(V2 / 'eventos_2025_verificados.csv')
resueltos = set(eventos['ticker_viejo'])
bc = bc[~bc['ticker'].isin(resueltos)].copy()
bc = bc[bc['cik'].notna()].copy()
bc['cik'] = bc['cik'].astype(float).astype(int)
print(len(bc), 'candidatos con cik por clasificar (sin cik quedan a manual)')

# Paso 1: el CIK sigue vivo en el archivo de la SEC?
sec_por_cik = sec.drop_duplicates('cik').set_index('cik')['ticker']
bc['ticker_sec_hoy'] = bc['cik'].map(sec_por_cik)
bc['clase'] = np.select(
    [bc['ticker_sec_hoy'].isna(),
     bc['ticker_sec_hoy'].str.split(r'[.\-$]').str[0] != bc['ticker']],
    ['baja_probable', 'renombre_detectado'],
    default='revisar_mismo_ticker')
print(bc['clase'].value_counts().to_string())

# Paso 2: fechar las bajas probables con el Form 25 en EDGAR
CHECK25 = V2 / 'bajas_fechadas_edgar.csv'
hechos25 = set()
filas25 = []
if CHECK25.exists():
    prev = pd.read_csv(CHECK25)
    hechos25 = set(prev['cik'].astype(int))
    filas25 = prev.to_dict('records')

pend25 = bc[(bc['clase'] == 'baja_probable') & (~bc['cik'].isin(hechos25))]
print(len(pend25), 'CIKs por consultar para Form 25')

for i, row in enumerate(pend25.itertuples(), 1):
    cik = int(row.cik)
    fila = {'ticker': row.ticker, 'cik': cik, 'fecha_form25': None,
            'fecha_form15': None, 'ultimo_filing': None, 'error': ''}
    try:
        url = f'https://data.sec.gov/submissions/CIK{cik:010d}.json'
        j = requests.get(url, headers=UA, timeout=30).json()
        forms = j['filings']['recent']['form']
        fechas = j['filings']['recent']['filingDate']
        f25 = [d for f, d in zip(forms, fechas) if f.startswith('25')]
        f15 = [d for f, d in zip(forms, fechas) if f.startswith('15')]
        fila['fecha_form25'] = max(f25) if f25 else None
        fila['fecha_form15'] = max(f15) if f15 else None
        fila['ultimo_filing'] = max(fechas) if fechas else None
    except Exception as e:
        fila['error'] = str(e)[:100]
    filas25.append(fila)
    if i % 50 == 0 or i == len(pend25):
        pd.DataFrame(filas25).to_csv(CHECK25, index=False)
        print(f'{i}/{len(pend25)} guardado')
    time.sleep(0.15)

r25 = pd.DataFrame(filas25)
print('con Form 25:', r25['fecha_form25'].notna().sum(), 'de', len(r25))
bc.to_csv(V2 / 'bajas_clasificadas_2025_2026.csv', index=False)

492 candidatos con cik por clasificar (sin cik quedan a manual)
clase
baja_probable           300
renombre_detectado      141
revisar_mismo_ticker     51
0 CIKs por consultar para Form 25
con Form 25: 297 de 300


In [16]:
# Celda D6b - Que son los candidatos a baja SIN cik?
bc_all = pd.read_csv(sorted(V2.glob('candidatos_bajas_2025_2026_*.csv'))[-1])
sin_cik = bc_all[bc_all['cik'].isna() & ~bc_all['ticker'].isin(resueltos)]
print(len(sin_cik), 'sin cik')
print(sin_cik['shrcd'].value_counts().to_string())
print('acciones comunes (shrcd 10/11/12):', sin_cik['shrcd'].isin([10, 11, 12]).sum())

606 sin cik
shrcd
73    232
12    177
11     75
31     43
44     30
14     28
18     14
71      3
48      2
72      2
acciones comunes (shrcd 10/11/12): 252


In [28]:
# Celda D7 - Ensamble de la ver 03: padron 2020 - jun 2026 (v4 final)
W2 = pd.Timestamp('2026-06-30')
COLA0 = pd.Timestamp('2025-01-01')

vig = pd.read_csv(V2 / 'padron_vigencias_2020_2024_mkt_flags.csv',
                  keep_default_na=False, na_values=[''],
                  parse_dates=['fecha_inicio', 'fecha_fin', 'snapshot_mercado',
                               'dlstdt', 'delist_date', 'primera_fecha'])
eventos = pd.read_csv(V2 / 'eventos_2025_verificados.csv', parse_dates=['fecha_efectiva'])
ed  = pd.read_csv(V2 / 'altas_fechadas_edgar.csv',
                  parse_dates=['fecha_8a', 'primer_filing', 'fecha_alta_aprox'])
dep = pd.read_csv(V2 / 'altas_emisores_2025_2026_depurado.csv')
bc  = pd.read_csv(V2 / 'bajas_clasificadas_2025_2026.csv')
r25 = pd.read_csv(V2 / 'bajas_fechadas_edgar.csv',
                  parse_dates=['fecha_form25', 'fecha_form15', 'ultimo_filing'])

vig = vig.assign(fecha_estimada=False, pendiente_clasificar=False,
                 fuente_cola='', ticker_previo=vig['ticker_previo'].fillna(''),
                 ticker_siguiente=vig['ticker_siguiente'].fillna(''))

# MultiIndex (permno, bloque) para editar vidas por llave
vig = vig.set_index(['permno', 'bloque'], drop=False)
vig.index.names = ['permno_idx', 'bloque_idx']

def base_sym(s):
    return re.split(r'[.\-$+=/ ]', str(s))[0] if pd.notna(s) else ''

# --- 1. Los 24 eventos verificados (fechas con fuente primaria) ---
nuevas = []
for ev in eventos.itertuples():
    mask = (vig['ticker'] == ev.ticker_viejo) & (vig['estado_vida'] == 'active')
    if mask.sum() != 1:
        print('AVISO evento', ev.ticker_viejo, '-> vidas encontradas:', mask.sum())
        continue
    idx = vig.index[mask][0]
    if ev.tipo == 'baja':
        vig.loc[idx, ['fecha_fin', 'delist_date']] = [ev.fecha_efectiva, ev.fecha_efectiva]
        vig.loc[idx, ['listing_status', 'estado_vida', 'fuente_cola']] = \
            ['delisted', 'delisted', 'evento_verificado']
    else:
        vig.loc[idx, 'fecha_fin'] = ev.fecha_efectiva - pd.Timedelta(days=1)
        vig.loc[idx, ['estado_vida', 'ticker_siguiente', 'fuente_cola']] = \
            ['renombrada', ev.ticker_nuevo, 'evento_verificado']
        base = vig.loc[idx]
        nuevas.append({
            'permno': base['permno'], 'bloque': int(base['bloque']) + 100,
            'ticker': ev.ticker_nuevo, 'comnam': str(ev.empresa).upper(),
            'fecha_inicio': ev.fecha_efectiva, 'fecha_fin': W2,
            'shrcd': base['shrcd'], 'exchcd': base['exchcd'], 'permco': base['permco'],
            'gvkey': base['gvkey'], 'cik': base['cik'], 'gsector': base['gsector'],
            'listing_status': 'active', 'estado_vida': 'active',
            'ticker_previo': ev.ticker_viejo, 'ticker_siguiente': '',
            'fecha_estimada': False, 'pendiente_clasificar': False,
            'fuente_cola': 'evento_verificado',
        })

# --- 2. Bajas probables con fecha de EDGAR (Form 25 / Form 15) ---
# Reglas: (a) un Form 25 anterior a 2025 es una mudanza de bolsa vieja, no la
# baja actual; (b) un Form 25 posterior al 30-jun-2026 es baja fuera de la
# ventana: la vida sigue activa dentro del estudio
r25 = r25.assign(fecha_baja=r25['fecha_form25'].fillna(r25['fecha_form15']))
fechas_baja = r25.dropna(subset=['fecha_baja']).set_index('cik')['fecha_baja']
for row in bc[bc['clase'] == 'baja_probable'].itertuples():
    idx = (row.permno, row.bloque)
    if idx not in vig.index:
        continue
    fb = fechas_baja.get(int(row.cik), pd.NaT)
    if pd.notna(fb) and COLA0 <= fb <= W2:
        vig.loc[idx, ['fecha_fin', 'delist_date']] = [fb, fb]
        vig.loc[idx, ['listing_status', 'estado_vida', 'fuente_cola']] = \
            ['delisted', 'delisted', 'edgar_form25']
    elif pd.notna(fb) and fb > W2:
        vig.loc[idx, 'fuente_cola'] = 'baja_post_ventana'
    else:
        vig.loc[idx, ['estado_vida', 'pendiente_clasificar', 'fuente_cola']] = \
            ['no_listado_2026', True, 'sin_fecha_baja']

# --- 3. Renombres detectados por CIK (sucesor conocido, fecha pendiente) ---
suc_de = {}
for row in bc[bc['clase'] == 'renombre_detectado'].itertuples():
    idx = (row.permno, row.bloque)
    if idx not in vig.index:
        continue
    suc = base_sym(row.ticker_sec_hoy)
    vig.loc[idx, ['estado_vida', 'ticker_siguiente', 'fecha_estimada', 'fuente_cola']] = \
        ['renombrada', suc, True, 'renombre_detectado_sin_fecha']
    suc_de[suc] = row.ticker

# --- 4. Pendientes: revisar_mismo_ticker + candidatos sin cik ---
for row in bc[bc['clase'] == 'revisar_mismo_ticker'].itertuples():
    idx = (row.permno, row.bloque)
    if idx in vig.index:
        vig.loc[idx, ['estado_vida', 'pendiente_clasificar', 'fuente_cola']] = \
            ['no_listado_2026', True, 'revisar_mismo_ticker']

bc_all = pd.read_csv(sorted(V2.glob('candidatos_bajas_2025_2026_*.csv'))[-1])
resueltos = set(eventos['ticker_viejo'])
sin_cik = bc_all[bc_all['cik'].isna() & ~bc_all['ticker'].isin(resueltos)]
for row in sin_cik.itertuples():
    idx = (row.permno, row.bloque)
    if idx in vig.index:
        vig.loc[idx, ['estado_vida', 'pendiente_clasificar', 'fuente_cola']] = \
            ['no_listado_2026', True, 'sin_cik']

# --- 5. Extender a jun-2026 las vidas que siguen activas ---
abiertas = vig['estado_vida'] == 'active'
vig.loc[abiertas, 'fecha_fin'] = W2
print('vidas extendidas a 2026-06-30:', abiertas.sum())

# --- 6. Altas 2025 - jun 2026 ---
ed2 = ed.merge(dep[['raiz', 'symbol_raw', 'security_name', 'exchange']],
               left_on='ticker', right_on='raiz', how='left')
for row in ed2.itertuples():
    if isinstance(row.ticker, str) and '$' in row.ticker:
        continue   # instrumentos de deuda que se colaron (MER$K)
    # Solo los metodos EDGAR confiables aportan fecha; el resto arranca
    # al inicio de la cola con bandera de fecha estimada
    if row.metodo_fecha in ('edgar_8a', 'edgar_primer_filing') and pd.notna(row.fecha_alta_aprox):
        fecha = row.fecha_alta_aprox
    else:
        fecha = COLA0
    if fecha > W2:
        continue   # listada despues del 30-jun-2026: fuera de la ventana de estudio
    nombre = str(row.security_name).split(' - ')[0].upper() if pd.notna(row.security_name) else ''
    nuevas.append({
        'permno': np.nan, 'bloque': 1,
        'ticker': row.ticker, 'comnam': nombre,
        'fecha_inicio': fecha, 'fecha_fin': W2,
        'shrcd': np.nan, 'exchcd': np.nan, 'permco': np.nan,
        'gvkey': '', 'cik': row.cik, 'gsector': np.nan,
        'listing_status': 'active', 'estado_vida': 'active',
        'ticker_previo': suc_de.get(row.ticker, ''), 'ticker_siguiente': '',
        'fecha_estimada': pd.isna(row.fecha_alta_aprox) or row.metodo_fecha != 'edgar_8a',
        'pendiente_clasificar': False,
        'fuente_cola': str(row.metodo_fecha),
    })

v3 = pd.concat([vig.reset_index(drop=True), pd.DataFrame(nuevas)], ignore_index=True)
print(len(vig), 'vidas CRSP +', len(nuevas), 'de la cola =', len(v3))
print(v3['estado_vida'].value_counts().to_string())

vidas extendidas a 2026-06-30: 7782
12961 vidas CRSP + 939 de la cola = 13900
estado_vida
active             8721
delisted           3293
renombrada         1223
no_listado_2026     663


In [29]:
# Celda D8 - Diagnostico, checklist, validacion y exportacion de la ver 03

# --- Diagnostico de vidas invertidas (debe dar 0) ---
inv = v3[v3['fecha_inicio'] > v3['fecha_fin']]
print(len(inv), 'vidas invertidas')
if len(inv):
    print(inv[['ticker', 'comnam', 'fecha_inicio', 'fecha_fin',
               'estado_vida', 'fuente_cola']].head(20).to_string(index=False))
    print('\nHay invertidas: NO se exporta. Pasar esta tabla a Claude.')
else:
    # --- Checklist de anclas ampliado ---
    anclas = ['GME', 'FB', 'META', 'BBBY', 'SMCI', 'SQ', 'XYZ', 'BGNE', 'ONC',
              'NEP', 'XIFR', 'PLYA', 'MRIN', 'ABTC', 'FISV', 'FI']
    cols = ['ticker', 'permno', 'comnam', 'fecha_inicio', 'fecha_fin',
            'estado_vida', 'ticker_previo', 'ticker_siguiente',
            'fecha_estimada', 'fuente_cola']
    print(v3[v3['ticker'].isin(anclas)][cols]
          .sort_values(['ticker', 'fecha_inicio']).to_string(index=False))

    # --- Validacion basica ---
    assert (v3['fecha_inicio'] <= v3['fecha_fin']).all(), 'fechas invertidas'
    assert v3['fecha_fin'].max() <= W2, 'fechas fuera de ventana'
    dup = v3.duplicated(['ticker', 'fecha_inicio', 'fecha_fin']).sum()
    print('\nduplicados exactos de vida:', dup)
    print('con fecha estimada:', v3['fecha_estimada'].sum(),
          '| pendientes de clasificar:', v3['pendiente_clasificar'].sum())

    # --- Exportacion ---
    v3.to_csv(V2 / 'padron_vigencias_2020_2026_ver03.csv', index=False)
    print(len(v3), 'vidas |', len(v3.columns),
          'columnas -> padron_vigencias_2020_2026_ver03.csv')

0 vidas invertidas
ticker  permno                                                 comnam fecha_inicio  fecha_fin estado_vida ticker_previo ticker_siguiente  fecha_estimada                  fuente_cola
  ABTC     NaN                                 AMERICAN BITCOIN CORP.   2025-01-01 2026-06-30      active                                           True                       manual
  BBBY 77659.0                                  BED BATH & BEYOND INC   2020-01-01 2023-05-02    delisted                                          False                             
  BBBY     NaN                   BED BATH & BEYOND, INC. COMMON STOCK   2025-01-01 2026-06-30      active          BYON                             True             revisar_8a_viejo
  BGNE 15936.0                                            BEIGENE LTD   2020-01-01 2025-01-01  renombrada                            ONC           False            evento_verificado
    FB 13407.0                                     META PLATFORMS INC  